# CrossDocked JSD tables

The numbers behind `figures/fig-jsd/`, as tables to read and as LaTeX to paste.

| | |
|---|---|
| **data** | `results/task2-drugdesign/_shared/260913_crossdocked_jsd/crossdocked_jsd.json`, written by `bash voxbind/scripts/89_eval_crossdocked_jsd.sh` (~15 s, no GPU). Re-run it first if the samples changed. |
| **reference (main)** | every CrossDocked2020 ligand of the train+test split (`split_by_name.pt`, 100,100 files), with **each distinct molecule weighing one**. The 100,000 training files hold only ~8.8k molecules, because one ligand is cross-docked into up to 869 pockets. |
| **reference (appendix)** | the 100 **test** ligands, the protocol behind TargetDiff Table 2/3 and VoxBind Table 2. The script's self-check reproduces TargetDiff's published row against them (printed below). |
| **generated** | every arm over the same 79 pockets that have experimental density; disconnected molecules dropped. |
| **JSD** | `scipy.spatial.distance.jensenshannon`, which is the JS *distance* (√divergence). Both papers print this number and call it a divergence, so the captions keep their wording. |

**Why the main reference is not the test set.** 100 ligands are too few to estimate a bond-length distribution. A model that sampled the training distribution *exactly* would still score C=N JSD 0.52 and C=C 0.46 against them (16 and 40 bonds), which is most of every method's score in those columns. The rankings agree either way (the sensitivity table below), but the test reference compresses the differences, and it reorders atom type and C–C pairs.

Tables: **1** bond-distance JSD · **2** distribution summary · **3** ring sizes · **4** ring statistics · **A1/A2** the same as 1/2 against the test set · sensitivity of the ranking to the reference · **5** re-evaluation vs published (check only).

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo(start=None):
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "figures" / "draw.py").exists() and (p / "voxbind").is_dir():
            return p
    raise FileNotFoundError("open this notebook inside the VoxBind checkout, or set VOXBIND_REPO")


REPO = Path(os.environ["VOXBIND_REPO"]) if "VOXBIND_REPO" in os.environ else find_repo()
JSD_JSON = REPO / "results/task2-drugdesign/_shared/260913_crossdocked_jsd/crossdocked_jsd.json"
if not JSD_JSON.exists():
    raise FileNotFoundError(f"{JSD_JSON} is missing: run  bash voxbind/scripts/89_eval_crossdocked_jsd.sh")

D = json.load(open(JSD_JSON))
if "reference_test" not in D:
    raise ValueError("the JSON predates the train+test reference: re-run voxbind/scripts/89_eval_crossdocked_jsd.sh")
REF, REF_TEST, ARMS = D["reference"], D["reference_test"], D["arms"]
POCKETS = "/".join(str(n) for n in sorted({a["n_pockets"] for a in ARMS.values()}))
SELFCHECK = D.get("selfcheck")

print(JSD_JSON.relative_to(REPO))
print(f"main reference: CrossDocked train+test, {REF['n_files']:,} ligand files = "
      f"{REF['n_unique_ligands']:,} distinct molecules, each weighing 1 (up to {REF['max_poses_per_ligand']} poses of one)")
print(f"appendix reference: the {REF_TEST['n_mols']} CrossDocked test ligands · {len(ARMS)} arms over {POCKETS} pockets")
if SELFCHECK:
    print(f"self-check, TargetDiff over 100 pockets vs its published row (test reference): "
          f"max |Δ| bond JSD {SELFCHECK['max_abs_diff_bond_jsd']:.4f}, "
          f"ring % {SELFCHECK['max_abs_diff_ring_pct']:.2f} -> {'PASS' if SELFCHECK['pass'] else 'FAIL'}")
    shipped = SELFCHECK.get("shipped_histograms_vs_pose_weighted_bond_jsd")
    if shipped:
        print("TargetDiff's shipped training histograms vs our pose-weighted train+test, bond JSD: "
              + " ".join(f"{k} {v:.3f}" for k, v in shipped.items()))

results/task2-drugdesign/_shared/260913_crossdocked_jsd/crossdocked_jsd.json
main reference: CrossDocked train+test, 100,100 ligand files = 8,829 distinct molecules, each weighing 1 (up to 869 poses of one)
appendix reference: the 100 CrossDocked test ligands · 8 arms over 79 pockets
self-check, TargetDiff over 100 pockets vs its published row (test reference): max |Δ| bond JSD 0.0053, ring % 1.05 -> PASS
TargetDiff's shipped training histograms vs our pose-weighted train+test, bond JSD: C-C 0.017 C=C 0.038 C-N 0.017 C=N 0.043 C-O 0.020 C=O 0.023 C:C 0.013 C:N 0.021


## Settings

Row order, row names and the `\midrule` before our method follow `notebook/html/260827/table_drug_design.tex`, so these tables sit beside the drug-design table without re-ordering. Change a LaTeX name here, once, e.g. to `\textsc{CoDE}`.

In [2]:
# (JSON label, name shown here, LaTeX name, group). A change of group draws a \midrule;
# a label the JSON does not hold is skipped.
ROWS = [
    ("AR",         "AR",                        r"AR",                                                   "baseline"),
    ("Pocket2Mol", "Pocket2Mol",                r"Pocket2Mol",                                           "baseline"),
    ("DiffSBDD",   "DiffSBDD",                  r"DiffSBDD",                                             "baseline"),
    ("TargetDiff", "TargetDiff",                r"TargetDiff",                                           "baseline"),
    ("DecompDiff", "DecompDiff (ref-informed)", r"DecompDiff\textsubscript{\scriptsize ref-informed}",   "baseline"),
    ("VoxBind",    "VoxBind σ=0.9",             r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",      "baseline"),
    ("FuncBind",   "FuncBind",                  r"FuncBind",                                             "baseline"),
    ("CoDE",       "VoxBind + CDG",             r"\textbf{VoxBind + CDG}",                               "ours"),
]
DECIMALS = 3               # JSD cells
PCT_DECIMALS = 1           # percentage cells
DROP_LEADING_ZERO = False  # True prints 0.372 as .372, as the VoxBind paper does
MARK_BEST = True           # bold the best value, underline the second best
RING_TRANSPOSE = True      # Table 3 with ring sizes as rows and methods as columns, as TargetDiff prints it

rows = [r for r in ROWS if r[0] in ARMS]
unlisted = [lab for lab in ARMS if lab not in {r[0] for r in ROWS}]
print("tabulated:", ", ".join(name for _, name, _, _ in rows))
if unlisted:
    print("in the JSON but not in ROWS, so not tabulated:", ", ".join(unlisted))

tabulated: AR, Pocket2Mol, DiffSBDD, TargetDiff, DecompDiff (ref-informed), VoxBind σ=0.9, FuncBind, VoxBind + CDG


In [3]:
LATEX = {}  # table name -> LaTeX; the last cell can write them all to a .tex file
BOND_TYPES = D["published"]["bond_jsd_columns"]  # C-C C=C C-N C=N C-O C=O C:C C:N
SUMMARY = [("Bond (avg.)",    r"Bond (avg.)",    lambda j: j["bond_mean"]),
           ("All-atom pairs", r"All-atom pairs", lambda j: j["pair"]["All_12A"]),
           ("C–C pairs",      r"C--C pairs",     lambda j: j["pair"]["CC_2A"]),
           ("Atom type",      r"Atom type",      lambda j: j["atom_type"])]
REF_MAIN_TEX = (f"The reference is every ligand of the CrossDocked2020 training and test split "
                f"({REF['n_files']:,} ligand files, {REF['n_unique_ligands']:,} distinct molecules), each distinct "
                f"molecule weighing one; generated molecules come from the {POCKETS} test pockets with experimental "
                f"density data. ")
REF_TEST_TEX = (f"The reference is the {REF_TEST['n_mols']} CrossDocked2020 test ligands, as in TargetDiff and "
                f"VoxBind; generated molecules come from the {POCKETS} test pockets with experimental density data. ")


def _nd(nd, j):
    return nd[j] if isinstance(nd, (list, tuple)) else nd


def fmt(v, nd):
    # A cell as text. A missing value (an arm with no bond of that type) is '---', never 0.
    if v is None or pd.isna(v):
        return "---"
    if nd == 0:
        return f"{v:,.0f}"
    s = f"{v:.{nd}f}"
    if DROP_LEADING_ZERO and s.startswith("0."):
        s = s[1:]
    return s


def ranks(values, nd, lower_is_better=True):
    # {row: 'best' | 'second'}, decided on the values AS PRINTED, so a tie in the table is a tie here.
    printed = [None if v is None or pd.isna(v) else round(float(v), nd) for v in values]
    order = sorted({v for v in printed if v is not None}, reverse=not lower_is_better)
    out = {}
    for i, v in enumerate(printed):
        if v is None:
            continue
        if v == order[0]:
            out[i] = "best"
        elif len(order) > 1 and v == order[1]:
            out[i] = "second"
    return out


def show(df, nd, lower_cols=()):
    # The table in the notebook, best bold and second best underlined where lower is better.
    col_nd = lambda c: nd[c] if isinstance(nd, dict) else nd

    def mark(col):
        if not MARK_BEST or col.name not in lower_cols:
            return [""] * len(col)
        r = ranks(list(col), col_nd(col.name))
        return ["font-weight: bold" if r.get(i) == "best" else
                "text-decoration: underline" if r.get(i) == "second" else "" for i in range(len(col))]

    display(df.style.format({c: (lambda v, c=c: fmt(v, col_nd(c))) for c in df.columns}).apply(mark, axis=0))


def latex_table(columns, body, *, caption, label, first_col="Method", nd=3, lower=None,
                head_rows=(), foot_rows=()):
    # booktabs, in the layout of table_drug_design.tex.
    #   columns    LaTeX column headers
    #   body       [(LaTeX row name, group, [values])]; a change of group draws a \midrule
    #   lower      per column: True lower is better, False higher is better, None not ranked
    #   head_rows  [(LaTeX row name, [cell strings])] above the body, then a \midrule
    #   foot_rows  the same, below the body after a \midrule
    ncol = len(columns)
    marks = [{} for _ in range(ncol)]
    if MARK_BEST and lower is not None:
        for j, lower_is_better in enumerate(lower):
            if lower_is_better is not None:
                marks[j] = ranks([vals[j] for _, _, vals in body], _nd(nd, j), lower_is_better)

    def cell(i, j, v):
        s = fmt(v, _nd(nd, j))
        r = marks[j].get(i)
        return rf"\textbf{{{s}}}" if r == "best" else rf"\underline{{{s}}}" if r == "second" else s

    def row(cells):
        return "            " + " & ".join(cells) + r" \\"

    out = [r"\begin{table}[!t]", r"    \centering", r"    \caption{", f"        {caption}", r"    }",
           rf"    \label{{{label}}}", r"    \resizebox{.98\textwidth}{!}{%",
           r"        \begin{tabular}{@{}l" + "c" * ncol + r"@{}}", r"            \toprule",
           row([rf"\textbf{{{first_col}}}"] + [rf"\textbf{{{c}}}" for c in columns]),
           r"            \midrule"]
    for name, cells in head_rows:
        out.append(row([name] + list(cells)))
    if head_rows:
        out.append(r"            \midrule")
    prev = None
    for i, (name, group, vals) in enumerate(body):
        if prev is not None and group != prev:
            out.append(r"            \midrule")
        prev = group
        out.append(row([name] + [cell(i, j, v) for j, v in enumerate(vals)]))
    if foot_rows:
        out.append(r"            \midrule")
        for name, cells in foot_rows:
            out.append(row([name] + list(cells)))
    out += [r"            \bottomrule", r"        \end{tabular}", r"    }", r"\end{table}"]
    return "\n".join(out)


def bond_df(key):
    # Bond JSD per type for every tabulated arm, against the reference `key` names.
    df = pd.DataFrame({t: [ARMS[lab][key]["bond"][t] for lab, *_ in rows] for t in BOND_TYPES},
                      index=pd.Index([name for _, name, *_ in rows], name="Method"))
    df["Avg."] = [ARMS[lab][key]["bond_mean"] for lab, *_ in rows]
    return df


def bond_latex(df, *, caption, label, foot_rows=()):
    return latex_table([t.replace("-", "--") for t in BOND_TYPES] + ["Avg."],
                       [(tex, group, list(df.loc[name])) for _, name, tex, group in rows],
                       lower=[True] * (len(BOND_TYPES) + 1), nd=DECIMALS, label=label,
                       caption=caption, foot_rows=foot_rows)


def summary_df(key):
    df = pd.DataFrame({c: [get(ARMS[lab][key]) for lab, *_ in rows] for c, _, get in SUMMARY},
                      index=pd.Index([name for _, name, *_ in rows], name="Method"))
    df["Molecules"] = [ARMS[lab]["n_mols"] for lab, *_ in rows]
    return df


def summary_latex(df, *, caption, label):
    return latex_table([tex for _, tex, _ in SUMMARY] + [r"\# Mols"],
                       [(tex, group, list(df.loc[name])) for _, name, tex, group in rows],
                       lower=[True] * len(SUMMARY) + [None], nd=[DECIMALS] * len(SUMMARY) + [0],
                       label=label, caption=caption)


SUMMARY_ND = {**{c: DECIMALS for c, _, _ in SUMMARY}, "Molecules": 0}
BOND_CAPTION = (r"``--'', ``='' and ``:'' denote single, double and aromatic bonds; Avg.\ is the mean over the "
                r"eight bond types. The best value is in bold and the second best underlined.")
SUMMARY_CAPTION = (r"Bond: mean bond-distance JSD over eight bond types. All-atom and C--C pairs: distances between "
                   r"heavy atoms of the same molecule, under 12\,\AA{} and under 2\,\AA{}. Atom type: heavy-atom "
                   r"element shares over C, N, O, F, P, S and Cl. \# Mols counts the single-component molecules "
                   r"scored. The best value is in bold and the second best underlined.")

## Table 1: bond-distance JSD

Same columns as VoxBind Table 2 (and TargetDiff Table 3), plus the average over the eight bond types, against the train+test reference.

In [4]:
bond = bond_df("jsd")
show(bond, DECIMALS, lower_cols=list(bond.columns))
print()
LATEX["bond"] = bond_latex(
    bond, label="tab:jsd-bond",
    caption=(r"\textbf{Jensen--Shannon divergence ($\downarrow$) between the bond-distance distributions of "
             r"reference and generated molecules.} " + REF_MAIN_TEX + BOND_CAPTION))
print(LATEX["bond"])

,C-C,C=C,C-N,C=N,C-O,C=O,C:C,C:N,Avg.
Method,,,,,,,,,
AR,0.575,0.417,0.384,0.437,0.399,0.508,0.459,0.452,0.454
Pocket2Mol,0.433,0.316,0.312,0.357,0.311,0.454,0.426,0.399,0.376
DiffSBDD,0.349,0.285,0.306,0.318,0.331,0.378,0.327,0.266,0.320
TargetDiff,0.302,0.186,0.238,0.155,0.281,0.398,0.194,0.140,0.237
DecompDiff (ref-informed),0.259,0.279,0.204,0.287,0.219,0.313,0.188,0.112,0.233
VoxBind σ=0.9,0.237,0.307,0.210,0.116,0.205,0.203,0.154,0.131,0.195
FuncBind,0.465,0.427,0.370,0.410,0.439,0.491,0.524,0.472,0.450
VoxBind + CDG,0.320,0.300,0.242,0.126,0.279,0.249,0.156,0.106,0.222



\begin{table}[!t]
    \centering
    \caption{
        \textbf{Jensen--Shannon divergence ($\downarrow$) between the bond-distance distributions of reference and generated molecules.} The reference is every ligand of the CrossDocked2020 training and test split (100,100 ligand files, 8,829 distinct molecules), each distinct molecule weighing one; generated molecules come from the 79 test pockets with experimental density data. ``--'', ``='' and ``:'' denote single, double and aromatic bonds; Avg.\ is the mean over the eight bond types. The best value is in bold and the second best underlined.
    }
    \label{tab:jsd-bond}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}lccccccccc@{}}
            \toprule
            \textbf{Method} & \textbf{C--C} & \textbf{C=C} & \textbf{C--N} & \textbf{C=N} & \textbf{C--O} & \textbf{C=O} & \textbf{C:C} & \textbf{C:N} & \textbf{Avg.} \\
            \midrule
            AR & 0.575 & 0.417 & 0.384 & 0.437 & 0.399 & 0.508 & 0.459 & 0.452 &

## Table 2: distribution summary

Mean bond-distance JSD, heavy-atom pair distances (all pairs under 12 Å, C–C pairs under 2 Å: TargetDiff Fig. 2) and element composition over C, N, O, F, P, S, Cl, against the train+test reference.

In [5]:
summary = summary_df("jsd")
show(summary, SUMMARY_ND, lower_cols=[c for c, _, _ in SUMMARY])
print()
LATEX["summary"] = summary_latex(
    summary, label="tab:jsd-summary",
    caption=(r"\textbf{Match to CrossDocked2020 ligands, Jensen--Shannon divergence ($\downarrow$).} "
             + REF_MAIN_TEX + SUMMARY_CAPTION))
print(LATEX["summary"])

,Bond (avg.),All-atom pairs,C–C pairs,Atom type,Molecules
Method,,,,,
AR,0.454,0.132,0.416,0.094,"7,655"
Pocket2Mol,0.376,0.114,0.359,0.078,"7,772"
DiffSBDD,0.320,0.094,0.332,0.041,"7,720"
TargetDiff,0.237,0.060,0.212,0.051,"7,287"
DecompDiff (ref-informed),0.233,0.054,0.197,0.050,"6,427"
VoxBind σ=0.9,0.195,0.033,0.127,0.063,"7,888"
FuncBind,0.450,0.101,0.438,0.082,"7,895"
VoxBind + CDG,0.222,0.049,0.186,0.062,"7,873"



\begin{table}[!t]
    \centering
    \caption{
        \textbf{Match to CrossDocked2020 ligands, Jensen--Shannon divergence ($\downarrow$).} The reference is every ligand of the CrossDocked2020 training and test split (100,100 ligand files, 8,829 distinct molecules), each distinct molecule weighing one; generated molecules come from the 79 test pockets with experimental density data. Bond: mean bond-distance JSD over eight bond types. All-atom and C--C pairs: distances between heavy atoms of the same molecule, under 12\,\AA{} and under 2\,\AA{}. Atom type: heavy-atom element shares over C, N, O, F, P, S and Cl. \# Mols counts the single-component molecules scored. The best value is in bold and the second best underlined.
    }
    \label{tab:jsd-summary}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}lccccc@{}}
            \toprule
            \textbf{Method} & \textbf{Bond (avg.)} & \textbf{All-atom pairs} & \textbf{C--C pairs} & \textbf{Atom type} & \textbf{\# Mols} \

## Table 3: ring sizes

The share of rings of each size among rings of size 3–9, as in TargetDiff Table 2 (whose columns sum to 100). Both references are shown: train+test, and the test set TargetDiff's reference column used. The notebook table also shows the share of *all* rings that are 10-membered or larger; the LaTeX leaves it out, as the published table does.

In [6]:
SIZES = D["published"]["ring_size_pct_columns"]  # "3" .. "9"
REF_SETS = [("Reference (train+test)", r"Ref.", r"Reference", REF),
            ("Reference (test)", r"Ref.\ (test)", r"Reference (test)", REF_TEST)]
sets = [(name, x) for name, _, _, x in REF_SETS] + [(name, ARMS[lab]) for lab, name, *_ in rows]
ring = pd.DataFrame({f"{s}-ring": [x["ring_size_pct"][s] for _, x in sets] for s in SIZES},
                    index=pd.Index([n for n, _ in sets], name="Method"))
ring["10+ (% of all rings)"] = [x["ring_size_pct"]["10+ of all"] for _, x in sets]
show(ring, PCT_DECIMALS)
print()

ring_caption = (r"\textbf{Percentage (\%) of rings of each size} among the rings of size 3--9, for the reference "
                r"ligands and generated molecules (as in TargetDiff, Table 2). "
                f"The references are every ligand of the CrossDocked2020 training and test split ({REF['n_unique_ligands']:,} "
                f"distinct molecules, each weighing one) and the {REF_TEST['n_mols']} test ligands; generated molecules "
                f"come from the {POCKETS} test pockets with experimental density data.")
if RING_TRANSPOSE:
    LATEX["ring_size"] = latex_table(
        [short for _, short, _, _ in REF_SETS] + [tex for _, _, tex, _ in rows],
        [(s, "size", [x["ring_size_pct"][s] for _, _, _, x in REF_SETS]
          + [ARMS[lab]["ring_size_pct"][s] for lab, *_ in rows]) for s in SIZES],
        first_col="Ring size", nd=PCT_DECIMALS, label="tab:ring-size", caption=ring_caption)
else:
    LATEX["ring_size"] = latex_table(
        [f"{s}-ring" for s in SIZES],
        [(tex, group, [ARMS[lab]["ring_size_pct"][s] for s in SIZES]) for lab, _, tex, group in rows],
        head_rows=[(long, [fmt(x["ring_size_pct"][s], PCT_DECIMALS) for s in SIZES]) for _, _, long, x in REF_SETS],
        nd=PCT_DECIMALS, label="tab:ring-size", caption=ring_caption)
print(LATEX["ring_size"])

,3-ring,4-ring,5-ring,6-ring,7-ring,8-ring,9-ring,10+ (% of all rings)
Method,,,,,,,,
Reference (train+test),1.5,0.3,27.3,70.0,0.8,0.1,0.0,0.6
Reference (test),1.7,0.0,30.2,67.4,0.8,0.0,0.0,2.0
AR,29.8,0.2,13.8,52.3,2.4,1.3,0.2,0.8
Pocket2Mol,0.1,0.0,16.8,78.0,4.3,0.7,0.1,0.6
DiffSBDD,17.3,2.9,27.7,41.7,8.2,1.7,0.5,1.3
TargetDiff,0.0,2.5,30.7,51.6,11.9,2.4,0.9,2.8
DecompDiff (ref-informed),2.7,3.9,35.6,43.8,11.2,2.2,0.6,1.5
VoxBind σ=0.9,0.0,0.2,20.0,78.8,0.8,0.1,0.0,2.3
FuncBind,0.0,0.4,16.1,82.9,0.4,0.1,0.1,0.2



\begin{table}[!t]
    \centering
    \caption{
        \textbf{Percentage (\%) of rings of each size} among the rings of size 3--9, for the reference ligands and generated molecules (as in TargetDiff, Table 2). The references are every ligand of the CrossDocked2020 training and test split (8,829 distinct molecules, each weighing one) and the 100 test ligands; generated molecules come from the 79 test pockets with experimental density data.
    }
    \label{tab:ring-size}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}lcccccccccc@{}}
            \toprule
            \textbf{Ring size} & \textbf{Ref.} & \textbf{Ref.\ (test)} & \textbf{AR} & \textbf{Pocket2Mol} & \textbf{DiffSBDD} & \textbf{TargetDiff} & \textbf{DecompDiff\textsubscript{\scriptsize ref-informed}} & \textbf{VoxBind\textsubscript{\scriptsize $\sigma$=0.9}} & \textbf{FuncBind} & \textbf{\textbf{VoxBind + CDG}} \\
            \midrule
            3 & 1.5 & 1.7 & 29.8 & 0.1 & 17.3 & 0.0 & 2.7 & 0.0 & 0.0 & 0.0 

## Table 4: ring statistics

All three quantities of VoxBind Fig. 10 in numbers: ring sizes are Table 3; here are **rings per molecule** (mean, ring-free share, share with four or more) and the **aromatic fraction**, computed exactly rather than read off the plotted bins. The aromatic fraction comes both as the paper defines it, the share of a molecule's heavy *atoms* (mean, and the share of molecules below 0.05), and as the share of its *rings* that are aromatic (mean over molecules with at least one ring). The figures are `figures/fig-jsd/rings-*.png` (the Fig. 10 layout), `n-rings-*.png` and `aromatic-*.png`. Nothing is ranked, because "better" here means closer to the reference.

In [7]:
def ring_stats(s):
    nr = {int(k): v for k, v in s["n_rings_counts"].items()}
    n, arom = sum(nr.values()), s["arom_frac_counts"]
    return [s["mean_n_rings"],
            100 * nr.get(0, 0) / n,
            100 * sum(v for k, v in nr.items() if k >= 4) / n,
            s["mean_arom_atom_frac"],
            100 * arom[0] / sum(arom),
            s["mean_arom_ring_frac"]]

STAT_COLS = [("Rings / mol.",                   r"Rings / mol.",                     2),
             ("No ring (%)",                    r"No ring (\%)",                     PCT_DECIMALS),
             ("≥4 rings (%)",                   r"$\geq$4 rings (\%)",               PCT_DECIMALS),
             ("Aromatic atom frac. (mean)",     r"Arom.\ atom frac.",                2),
             ("Aromatic atom frac. < 0.05 (%)", r"Arom.\ atom frac.\ $<$0.05 (\%)",  PCT_DECIMALS),
             ("Aromatic ring frac. (mean)",     r"Arom.\ ring frac.",                2)]
stats = pd.DataFrame([ring_stats(x) for _, x in sets], columns=[c for c, _, _ in STAT_COLS],
                     index=pd.Index([n for n, _ in sets], name="Method"))
show(stats, {c: d for c, _, d in STAT_COLS})
print()

LATEX["ring_stats"] = latex_table(
    [tex for _, tex, _ in STAT_COLS],
    [(tex, group, list(stats.loc[name])) for _, name, tex, group in rows],
    head_rows=[(long, [fmt(v, d) for v, (_, _, d) in zip(ring_stats(x), STAT_COLS)]) for _, _, long, x in REF_SETS],
    nd=[d for _, _, d in STAT_COLS], label="tab:ring-stats",
    caption=(r"\textbf{Ring statistics} of the reference ligands and generated molecules: mean number of rings per "
             r"molecule; share of molecules with no ring and with at least four rings; mean aromatic fraction of a "
             r"molecule's heavy atoms and share of molecules where it is below 0.05; and mean aromatic fraction of a "
             r"molecule's rings, over molecules with at least one ring. "
             f"The references are every ligand of the CrossDocked2020 training and test split ({REF['n_unique_ligands']:,} "
             f"distinct molecules, each weighing one) and the {REF_TEST['n_mols']} test ligands; generated molecules "
             f"come from the {POCKETS} test pockets with experimental density data."))
print(LATEX["ring_stats"])

,Rings / mol.,No ring (%),≥4 rings (%),Aromatic atom frac. (mean),Aromatic atom frac. < 0.05 (%),Aromatic ring frac. (mean)
Method,,,,,,
Reference (train+test),2.83,6.8,33.3,0.39,20.2,0.64
Reference (test),2.47,13.0,20.0,0.29,34.0,0.55
AR,2.80,9.3,31.7,0.14,65.7,0.18
Pocket2Mol,2.94,4.6,32.1,0.44,20.9,0.57
DiffSBDD,2.32,11.1,20.9,0.14,60.9,0.24
TargetDiff,2.60,9.6,28.1,0.13,61.5,0.21
DecompDiff (ref-informed),2.24,13.9,18.2,0.19,51.1,0.35
VoxBind σ=0.9,2.78,9.9,34.3,0.31,30.9,0.47
FuncBind,2.04,14.1,14.9,0.10,71.8,0.17



\begin{table}[!t]
    \centering
    \caption{
        \textbf{Ring statistics} of the reference ligands and generated molecules: mean number of rings per molecule; share of molecules with no ring and with at least four rings; mean aromatic fraction of a molecule's heavy atoms and share of molecules where it is below 0.05; and mean aromatic fraction of a molecule's rings, over molecules with at least one ring. The references are every ligand of the CrossDocked2020 training and test split (8,829 distinct molecules, each weighing one) and the 100 test ligands; generated molecules come from the 79 test pockets with experimental density data.
    }
    \label{tab:ring-stats}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}lcccccc@{}}
            \toprule
            \textbf{Method} & \textbf{Rings / mol.} & \textbf{No ring (\%)} & \textbf{$\geq$4 rings (\%)} & \textbf{Arom.\ atom frac.} & \textbf{Arom.\ atom frac.\ $<$0.05 (\%)} & \textbf{Arom.\ ring frac.} \\
            \m

## Appendix A1/A2: against the 100 test ligands (published protocol)

Tables 1 and 2 again, scored the way TargetDiff and VoxBind scored them, so the values sit on the published scale. A1's last row counts the test set's bonds of each type: the C=C and C=N columns rest on 40 and 16 of them, and a model sampling the training distribution exactly would still score about 0.46 and 0.52 there.

In [8]:
bond_test = bond_df("jsd_test")
show(bond_test, DECIMALS, lower_cols=list(bond_test.columns))
print("test-set bonds per type:", ", ".join(f"{t} {REF_TEST['bond_n'][t]}" for t in BOND_TYPES), "\n")
LATEX["bond_test"] = bond_latex(
    bond_test, label="tab:jsd-bond-test",
    foot_rows=[(r"\textit{\# ref.\ bonds}", [str(REF_TEST["bond_n"][t]) for t in BOND_TYPES] + [""])],
    caption=(r"\textbf{Jensen--Shannon divergence ($\downarrow$) between the bond-distance distributions of "
             r"reference and generated molecules, against the test set.} " + REF_TEST_TEX + BOND_CAPTION
             + r" The last row counts the reference bonds of each type."))
print(LATEX["bond_test"])

,C-C,C=C,C-N,C=N,C-O,C=O,C:C,C:N,Avg.
Method,,,,,,,,,
AR,0.612,0.599,0.468,0.621,0.499,0.561,0.453,0.529,0.543
Pocket2Mol,0.492,0.563,0.418,0.638,0.451,0.509,0.414,0.477,0.495
DiffSBDD,0.405,0.566,0.405,0.607,0.460,0.448,0.324,0.373,0.448
TargetDiff,0.370,0.501,0.361,0.545,0.416,0.466,0.258,0.238,0.394
DecompDiff (ref-informed),0.333,0.541,0.335,0.565,0.346,0.390,0.242,0.227,0.372
VoxBind σ=0.9,0.314,0.519,0.334,0.528,0.370,0.307,0.218,0.191,0.348
FuncBind,0.511,0.636,0.452,0.648,0.542,0.549,0.504,0.541,0.548
VoxBind + CDG,0.390,0.525,0.358,0.526,0.430,0.342,0.215,0.191,0.372


test-set bonds per type: C-C 716, C=C 40, C-N 245, C=N 16, C-O 336, C=O 101, C:C 500, C:N 213 

\begin{table}[!t]
    \centering
    \caption{
        \textbf{Jensen--Shannon divergence ($\downarrow$) between the bond-distance distributions of reference and generated molecules, against the test set.} The reference is the 100 CrossDocked2020 test ligands, as in TargetDiff and VoxBind; generated molecules come from the 79 test pockets with experimental density data. ``--'', ``='' and ``:'' denote single, double and aromatic bonds; Avg.\ is the mean over the eight bond types. The best value is in bold and the second best underlined. The last row counts the reference bonds of each type.
    }
    \label{tab:jsd-bond-test}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}lccccccccc@{}}
            \toprule
            \textbf{Method} & \textbf{C--C} & \textbf{C=C} & \textbf{C--N} & \textbf{C=N} & \textbf{C--O} & \textbf{C=O} & \textbf{C:C} & \textbf{C:N} & \textbf{Avg.} \\
    

In [9]:
summary_test = summary_df("jsd_test")
show(summary_test, SUMMARY_ND, lower_cols=[c for c, _, _ in SUMMARY])
print()
LATEX["summary_test"] = summary_latex(
    summary_test, label="tab:jsd-summary-test",
    caption=(r"\textbf{Match to the CrossDocked2020 test ligands, Jensen--Shannon divergence ($\downarrow$).} "
             + REF_TEST_TEX + SUMMARY_CAPTION))
print(LATEX["summary_test"])

,Bond (avg.),All-atom pairs,C–C pairs,Atom type,Molecules
Method,,,,,
AR,0.543,0.125,0.429,0.106,"7,655"
Pocket2Mol,0.495,0.118,0.385,0.115,"7,772"
DiffSBDD,0.448,0.084,0.325,0.082,"7,720"
TargetDiff,0.394,0.053,0.225,0.087,"7,287"
DecompDiff (ref-informed),0.372,0.044,0.207,0.043,"6,427"
VoxBind σ=0.9,0.348,0.059,0.207,0.111,"7,888"
FuncBind,0.548,0.095,0.436,0.083,"7,895"
VoxBind + CDG,0.372,0.074,0.266,0.119,"7,873"



\begin{table}[!t]
    \centering
    \caption{
        \textbf{Match to the CrossDocked2020 test ligands, Jensen--Shannon divergence ($\downarrow$).} The reference is the 100 CrossDocked2020 test ligands, as in TargetDiff and VoxBind; generated molecules come from the 79 test pockets with experimental density data. Bond: mean bond-distance JSD over eight bond types. All-atom and C--C pairs: distances between heavy atoms of the same molecule, under 12\,\AA{} and under 2\,\AA{}. Atom type: heavy-atom element shares over C, N, O, F, P, S and Cl. \# Mols counts the single-component molecules scored. The best value is in bold and the second best underlined.
    }
    \label{tab:jsd-summary-test}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}lccccc@{}}
            \toprule
            \textbf{Method} & \textbf{Bond (avg.)} & \textbf{All-atom pairs} & \textbf{C--C pairs} & \textbf{Atom type} & \textbf{\# Mols} \\
            \midrule
            AR & 0.543 & 0.125 & 0.429 & 0

## Sensitivity: does the reference change the ranking?

Every headline JSD against the three references the eval builds: train+test with each distinct molecule weighing one (main), the same files with every pose weighing one (the distribution training saw), and the 100 test ligands. The first two nearly coincide, so the de-duplication moves nothing. The test set shifts every value up and compresses the gaps; its ranks are what differ.

In [10]:
REFS = [("train+test", "jsd"), ("pose-weighted", "jsd_pose_weighted"), ("test", "jsd_test")]
cols = {}
for metric, _, get in SUMMARY:
    for ref_name, key in REFS:
        cols[(metric, ref_name)] = [get(ARMS[lab][key]) for lab, *_ in rows]
sens = pd.DataFrame(cols, index=pd.Index([name for _, name, *_ in rows], name="Method"))
display(sens.style.format(precision=DECIMALS))
rank = sens.rank(method="min").astype(int)
print("rank within each column (1 = closest to that reference):")
display(rank)

rank within each column (1 = closest to that reference):


Bond (avg.)                    All-atom pairs  \
                           train+test pose-weighted test     train+test   
Method                                                                    
AR                                  8             8    7              8   
Pocket2Mol                          6             6    6              7   
DiffSBDD                            5             5    5              5   
TargetDiff                          4             4    4              4   
DecompDiff (ref-informed)           3             3    3              3   
VoxBind σ=0.9                       1             1    1              1   
FuncBind                            7             7    8              6   
VoxBind + CDG                       2             2    2              2   

                                              C–C pairs                     \
                          pose-weighted test train+test pose-weighted test   
Method                                                                       
AR                                    8    8          7             7    7   
Pocket2Mol                            7    7          6             6    6   
DiffSBDD                              5    5          5             5    5   
TargetDiff                            3    2          4             4    3   
DecompDiff (ref-informed)             2    1          3             3    2   
VoxBind σ=0.9                         1    3          1             1    1   
FuncBind                              6    6          8             8    8   
VoxBind + CDG                         4    4          2             2    4   

                           Atom type                     
                          train+test pose-weighted test  
Method                                                   
AR                                 8             8    5  
Pocket2Mol                         6             7    7  
DiffSBDD                           1             2    2  
TargetDiff                         3             3    4  
DecompDiff (ref-informed)          2             1    1  
VoxBind σ=0.9                      5             5    6  
FuncBind                           7             6    3  
VoxBind + CDG                      4             4    8

## Table 5: re-evaluation vs published (check only)

Our bond JSDs against the test set, next to the published ones, for the methods both cover. They are **not** expected to be equal: the published rows use all 100 pockets and each paper's own sample sets, while this eval uses the 79 density pockets. The like-for-like check is the TargetDiff self-check row, TargetDiff's samples over all 100 pockets.

In [11]:
PUB = D["published"]["bond_jsd"]
PUB_KEY = {"AR": "AR", "Pocket2Mol": "Pocket2Mol", "TargetDiff": "TargetDiff",
           "DecompDiff": "DecompDiff", "VoxBind": "VoxBind σ=0.9"}
recs = []
for lab, name, *_ in rows:
    if PUB_KEY.get(lab) in PUB:
        recs.append((name, f"this eval ({POCKETS} pockets)", *[ARMS[lab]["jsd_test"]["bond"][t] for t in BOND_TYPES]))
        recs.append((name, "published (100 pockets)", *PUB[PUB_KEY[lab]]))
if SELFCHECK:
    recs.append(("TargetDiff", "self-check (100 pockets)", *[SELFCHECK["bond_jsd"][t] for t in BOND_TYPES]))
check = pd.DataFrame(recs, columns=["Method", "Source", *BOND_TYPES]).set_index(["Method", "Source"])
display(check.style.format(precision=DECIMALS))

In [12]:
SAVE_TEX = False  # True writes every LaTeX table above into jsd-tables.tex beside this notebook
if SAVE_TEX:
    path = REPO / "figures" / "fig-jsd" / "jsd-tables.tex"
    path.write_text("\n\n".join(LATEX.values()) + "\n")
    print("wrote", path)
else:
    print(f"{len(LATEX)} LaTeX tables in LATEX: {', '.join(LATEX)}. Set SAVE_TEX = True to write them to a .tex file.")

6 LaTeX tables in LATEX: bond, summary, ring_size, ring_stats, bond_test, summary_test. Set SAVE_TEX = True to write them to a .tex file.
